## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Packages.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import pathlib
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta
import subprocess

Download repo and data files.

In [ ]:
from src.data import configure_root

# Get root and insert to sys path
root = configure_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

Sample random sections of the genome, weighted by chromosome length and within the specified sequence length range.

In [2]:
from src.data import sample_from_fasta

# Data directory
data_dir = root / "data"
fasta_file = data_dir / "hg38.fa"
chrom_size_file = data_dir / "hg38.chrom.sizes"

# Set parameters
n_seqs = 100
min_length = 20
max_length = 500

# Get pretraining data
pretraining = sample_from_fasta(
    fasta_file = fasta_file,
    chrom_size_file = chrom_size_file,
    n_seqs = n_seqs,
    min_length = min_length,
    max_length = max_length,
    seed = 111
)

## Tokenizer

Simple function to generate random samples to mimic fasta samples.

In [29]:
def generate_seqs(min_length, max_length, num_seqs, seed = None):  
    rng = np.random.default_rng(seed)
    seqs = []

    for i in range(num_seqs):
        length = rng.choice(np.arange(min_length, max_length))
        seq = [rng.choice(["A", "C", "G", "T"]) for i in range(length)]
        seq = "".join(seq)
        seqs.append(seq)
        
    return seqs

Train BPE tokenizer.

In [ ]:
from src.tokenize import BPETokenizer


## Model architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Implement simple attention block.

https://medium.com/@heyamit10/implement-self-attention-and-cross-attention-in-pytorch-cfe17ab0b3ee

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        
        # Queries, keys, values
        self.q_map = nn.Linear(d_model, d_model)
        self.k_map = nn.Linear(d_model, d_model)
        self.v_map = nn.Linear(d_model, d_model)

    def forward(self, x):

        # Input x[B, L, D]
        B, L, D = x.shape
           
        # q[B, L, d_model], k[B, L, d_model], v[B, L, d_model]
        q = self.q_map(x)
        k = self.k_map(x)
        v = self.v_map(x)

        # a[B, L, L] (softmax over last dimension L from keys)
        a = torch.softmax(q @ k.transpose(-2, -1) / (self.d_model ** 0.5), dim = -1)

        # y[B, L, d_model]
        out = a @ v

        return out

Implement multi-head attention block.

In [ ]:
class MultiHeadAttention():
    def __init__(self, d_model, num_heads):
        super().__init__()

Implement sparse MultiHeadAttention.

In [ ]:
class SparseMultiHeadAttention():
    def __init__(self, d_model, num_heads):
        super().__init__()

Implement transformer encoder block.

In [ ]:
class TransformerEncoder():
    def __init__(self):
        super().__init__()

Implement sparse attention transformer block.

In [ ]:
class SparseTransformerEncoder():
    def __init__(self):
        super().__init__()

Things that will be implemented:
- Padding to fixed max sequence length (512?)
- Embedding module
- Masked token prediction objective
- Sparse attention transformer
- 

Maybe pre-train on approx. 500k-1mil sequences?

Input to model should be a list of token IDs.

In [ ]:
class SparseGLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        
    ):
        super().__init__()

        self.embed = nn.Embedding

        # Transformer layers?
        
    def forward(self, x):

        return


class GLM(nn.Module):
    def __init(
        self,
        vocab_size
    ):
        super().__init__()



## Training

Implement masked language modeling training objective.
- Batching [B, L, D]
    - Token right padding should be done to the max length in the batch* (computational efficient)
- Attention mask to ignore padding tokens
- Train weights in batches

Reference: https://huggingface.co/learn/llm-course/en/chapter6/5